In [63]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_extract, when, lit, udf, regexp_replace, lower, trim, translate, initcap, split
from pyspark.sql.types import FloatType, StringType, IntegerType
import re

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

spark = (
    SparkSession.builder
    .appName("AutoTec_Batch_Processing")
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .config("spark.mongodb.write.connection.uri", MONGO_URI)
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1")
    .getOrCreate()
)

df = (
    spark.read.format("mongodb")
    .option("database", "proyecto_bigdata")
    .option("collection", "Contenedor_Autos_Limpio")
    .load()
)

In [65]:
#Eliminacion columna antiguedad
df = df.drop("antiguedad_auto")

#eliminacion registros "premium"
df = df[df["tipo_marca"] != "premium"]

#eliminacion de registros de marca con poca variabilidad
#menos de 10 registros por marcas
df = df.filter(~df["marca"].isin([r["marca"] for r in df.groupBy("marca").count().filter("count < 10").collect()]))

#eliminacion ciudades de puerto montt hacia abajo
ciudades_sur = [
    "puerto montt", "castro", "ancud", "osorno", "valdivia", "puerto varas",
    "pucón", "temuco", "angol", "villarrica", "la unión", "lago ranco",
    "puerto aysén", "coihaique", "coyhaique", "puerto natales", "punta arenas"
]
df = df.filter(~lower(col("ciudad")).isin(ciudades_sur))

#eliminacion autos eléctricos y columna es_ecologico
df = df.drop("es_ecologico")
df = df.filter(col("combustible") != "electrico")


In [67]:
df.printSchema()

root
 |-- _id: string (nullable = true)
 |-- cat_combustible: integer (nullable = true)
 |-- categoria_precio: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- combustible: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- foto_url: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- kilometraje: double (nullable = true)
 |-- marca: string (nullable = true)
 |-- modelo: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- rango_kilometraje: string (nullable = true)
 |-- tipo_marca: string (nullable = true)
 |-- url: string (nullable = true)
 |-- uso_anual_estimado: double (nullable = true)
 |-- usuario: string (nullable = true)
 |-- year: integer (nullable = true)



In [66]:
print(df.count())

1685


In [69]:
from pyspark.sql import SparkSession
import os
from dotenv import load_dotenv

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

spark = (
    SparkSession.builder
    .appName("AutoTec_Batch_Processing")
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .config("spark.mongodb.write.connection.uri", MONGO_URI)
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1")
    .getOrCreate()
)

df.write \
    .format("mongodb") \
    .mode("append") \
    .option("database", "proyecto_bigdata") \
    .option("collection", "Contenedor_Autos_Limpio1") \
    .save()

print("Datos limpios cargados correctamente")

Datos limpios cargados correctamente
